[← GstreamerExp hub](../../index.html) · [README](../../README.md) · [Hypothesis catalog](../../docs/HYPOTHESES.md)

# H4 — SCReAM keeps more video watchable than GCC on tight 5G recordings

**Status:** `supported` · **Source:** Project-internal 5G-recording SCReAM / GCC experiment


## Claim

On 5G recordings squeezed to one-third of their original capacity, SCReAM delivers more decoded frames to the viewer than GCC does, while the camera actually sends fewer bytes — on the two harder recordings (handover and resource-block).

## Predictions

- `scream_delivers_at_least_25_percent_more_viewer_frames_than_gcc_on_both_harder_recordings`
- `on_both_harder_recordings_camera_sends_no_more_than_85_percent_of_what_gcc_sends`
- `viewer_receives_within_10_percent_of_gcc_bytes_on_both_harder_recordings`
- `scream_delivers_at_least_2x_viewer_frames_on_at_least_one_harder_recording`
- `on_the_high_capacity_recording_both_algorithms_deliver_nearly_all_frames`

## Verdict

| Outcome | Predicate |
|---|---|
| **Supported when all** | <code>scream_delivers_at_least_25_percent_more_viewer_frames_than_gcc_on_both_harder_recordings</code><br><code>on_both_harder_recordings_camera_sends_no_more_than_85_percent_of_what_gcc_sends</code><br><code>viewer_receives_within_10_percent_of_gcc_bytes_on_both_harder_recordings</code><br><code>scream_delivers_at_least_2x_viewer_frames_on_at_least_one_harder_recording</code> |
| **Refuted when any** | <code>any_threshold_violated_on_completed_records</code> |
| **Untested when any** | <code>required_records_missing</code><br><code>required_summaries_missing</code> |


## Main claim

SCReAM wins on the stressed HO/RB traces because its post-payload RTP pacing turns congestion-control decisions into lower camera egress, while GCC lowers its estimator target but this pipeline still emits multi-Mbit/s RTP that the trace cannot carry, leaving fewer complete frames decodable at the viewer.


## Supporting evidence

- Within each trace pair, the trace, video, codec, bitrate bounds, recovery settings, sink, workers, and run length are fixed; only congestion_control.algorithm varies.
- CQI is the high-capacity control and both controllers deliver nearly all frames, so the advantage appears only when HO/RB capacity actually stresses the stream.
- On HO, SCReAM delivers 2.74x GCC's viewer frames; on RB, it delivers 1.39x GCC's viewer frames.
- The advantage is not from sending more traffic; SCReAM uses 0.72x GCC's camera egress on HO and 0.64x on RB.
- GCC's target falls on the stressed traces, but its measured camera egress remains about 2.35 Mbit/s on HO and 2.32 Mbit/s on RB.


## Trace Characteristics

- CQI x0.33 is a high-capacity control trace with no sub-1 Mbit/s bins and 3.68 Mbit/s median capacity.
- HO x0.33 is a handover-like trace with sharp fades: 18.7% of 100 ms bins below 1 Mbit/s and 2.06 Mbit/s median capacity.
- RB x0.33 is a resource-block scarcity trace with 18.4% of 100 ms bins below 1 Mbit/s and 1.70 Mbit/s median capacity.


## Findings and Limitations

**Findings**

- On the high-capacity recording (CQI), both algorithms deliver nearly all frames; this recording acts as the easy control.
- On the handover recording, SCReAM delivers about 2.7x as many frames to the viewer as GCC, while the camera sends fewer bytes.
- On the resource-block recording, SCReAM delivers about 1.4x as many frames to the viewer as GCC, while the camera sends fewer bytes.
- GCC's target bitrate does drop on the tighter recordings, but in this pipeline that lower target does not turn into fewer bytes leaving the camera.
- The result is meaningful because the test video is now heavy enough to actually keep the recordings busy (the H3 calibration issue is fixed for this experiment).

**Limitations**

- One test video, fixed recording, one set of bitrate bounds. The CQI recording shows SCReAM has no visible frame-delivery edge when the link has enough room for both algorithms.
- The latency numbers still have a clock-skew artifact between aum and veda. H4's main claim is frame delivery and camera bytes, not latency.


## Figures

![Figure H4-1. Mean viewer-depayloaded frames by trace. Each grouped pair compares SCReAM and GCC under the same trace and workload; S/G labels show the SCReAM/GCC frame ratio.](results/h4_frame_delivery_by_trace.svg)

*Figure H4-1. Mean viewer-depayloaded frames by trace. Each grouped pair compares SCReAM and GCC under the same trace and workload; S/G labels show the SCReAM/GCC frame ratio.*

![Figure H4-2. Mean camera egress bitrate by trace. S/G labels show whether the delivered-frame improvement came from simply sending more traffic; GCC's lower estimator target on stressed traces is not the same as lower emitted RTP rate.](results/h4_camera_egress_by_trace.svg)

*Figure H4-2. Mean camera egress bitrate by trace. S/G labels show whether the delivered-frame improvement came from simply sending more traffic; GCC's lower estimator target on stressed traces is not the same as lower emitted RTP rate.*

![Figure H4-3a. CQI fixed-trace time series. Left: shared capacity and camera egress. Right: cumulative delivered frames.](results/h4_cqi_timeseries.svg)

*Figure H4-3a. CQI fixed-trace time series. Left: shared capacity and camera egress. Right: cumulative delivered frames.*

![Figure H4-3b. HO fixed-trace time series. The wide two-panel layout holds the trace fixed and shows where SCReAM's delivered-frame curve separates from GCC.](results/h4_ho_timeseries.svg)

*Figure H4-3b. HO fixed-trace time series. The wide two-panel layout holds the trace fixed and shows where SCReAM's delivered-frame curve separates from GCC.*

![Figure H4-3c. RB fixed-trace time series. The wide two-panel layout holds the trace fixed and shows the moderate SCReAM frame-delivery advantage.](results/h4_rb_timeseries.svg)

*Figure H4-3c. RB fixed-trace time series. The wide two-panel layout holds the trace fixed and shows the moderate SCReAM frame-delivery advantage.*


## Tables

### `h4_arm_summary`

| trace | experiment | network | algorithm | config_id | pass_runs | viewer_frames_mean | camera_wire_mean_kbps | viewer_wire_mean_kbps | encoded_mean_kbps | encoder_target_mean_kbps | latency_p95_mean_ms |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| CQI | scream-vs-gcc-mahimahi-5g-cqi-x0p33-snow-384x216 | mahimahi-5g-cqi-100ms-x0p33 | SCReAM | 81 | 3 | 903.7 | 2,861.2 | 2,483.1 | 2,878.6 | 3,267.6 | -81.57 |
| CQI | scream-vs-gcc-mahimahi-5g-cqi-x0p33-snow-384x216 | mahimahi-5g-cqi-100ms-x0p33 | GCC | 82 | 3 | 901.3 | 2,446.4 | 2,097.9 | 2,416.2 | 2,460.0 | -96.90 |
| HO | scream-vs-gcc-mahimahi-5g-ho-x0p33-snow-384x216 | mahimahi-5g-ho-100ms-x0p33 | SCReAM | 79 | 3 | 544.7 | 1,696.0 | 1,398.5 | 2,448.9 | 1,351.9 | -108.8 |
| HO | scream-vs-gcc-mahimahi-5g-ho-x0p33-snow-384x216 | mahimahi-5g-ho-100ms-x0p33 | GCC | 80 | 3 | 199.0 | 2,353.0 | 1,427.7 | 2,319.4 | 331.3 | -101.5 |
| RB | scream-vs-gcc-mahimahi-5g-rb-x0p33-snow-384x216 | mahimahi-5g-rb-100ms-x0p33 | SCReAM | 83 | 3 | 445.3 | 1,480.8 | 1,236.0 | 2,416.5 | 1,096.8 | -132.4 |
| RB | scream-vs-gcc-mahimahi-5g-rb-x0p33-snow-384x216 | mahimahi-5g-rb-100ms-x0p33 | GCC | 84 | 3 | 319.3 | 2,324.9 | 1,302.5 | 2,290.8 | 177.5 | -122.7 |

### `h4_trace_comparison`

| trace | network | scream_viewer_frames_mean | gcc_viewer_frames_mean | viewer_frame_ratio_scream_over_gcc | viewer_frame_delta | scream_camera_wire_mean_kbps | gcc_camera_wire_mean_kbps | camera_wire_ratio_scream_over_gcc | scream_encoder_target_mean_kbps | gcc_encoder_target_mean_kbps | gcc_target_to_camera_egress_ratio | scream_viewer_wire_mean_kbps | gcc_viewer_wire_mean_kbps | viewer_wire_delta_pct |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| CQI | mahimahi-5g-cqi-100ms-x0p33 | 903.7 | 901.3 | 1.00 | 2.33 | 2,861.2 | 2,446.4 | 1.17 | 3,267.6 | 2,460.0 | 1.01 | 2,483.1 | 2,097.9 | 15.51 |
| HO | mahimahi-5g-ho-100ms-x0p33 | 544.7 | 199.0 | 2.74 | 345.7 | 1,696.0 | 2,353.0 | 0.721 | 1,351.9 | 331.3 | 0.141 | 1,398.5 | 1,427.7 | 2.04 |
| RB | mahimahi-5g-rb-100ms-x0p33 | 445.3 | 319.3 | 1.39 | 126.0 | 1,480.8 | 2,324.9 | 0.637 | 1,096.8 | 177.5 | 0.076 | 1,236.0 | 1,302.5 | 5.10 |


## Experimental setup

### `scream-vs-gcc-mahimahi-5g-cqi-x0p33-snow-384x216`

SCReAM vs GCC on one fixed translated 5G Mahimahi CQI-like trace scaled to 33% capacity, using calibrated high-entropy 384x216x30 snow video. The trace is fixed; the comparison varies only the congestion controller.


**Configurations:** `81` (scream), `82` (gcc) · **Reps:** 3

Spec: `specs/experiments/scream-vs-gcc-mahimahi-5g-cqi-x0p33-snow-384x216.yaml` · Record: `runs/experiments/scream-vs-gcc-mahimahi-5g-cqi-x0p33-snow-384x216.json` · Run: `python3 experiment.py scream-vs-gcc-mahimahi-5g-cqi-x0p33-snow-384x216`

**Status:** 6 of 6 runs completed.

### `scream-vs-gcc-mahimahi-5g-ho-x0p33-snow-384x216`

SCReAM vs GCC on one fixed translated 5G Mahimahi handover-like trace scaled to 33% capacity, using calibrated high-entropy 384x216x30 snow video. The trace is fixed; the comparison varies only the congestion controller.


**Configurations:** `79` (scream), `80` (gcc) · **Reps:** 3

Spec: `specs/experiments/scream-vs-gcc-mahimahi-5g-ho-x0p33-snow-384x216.yaml` · Record: `runs/experiments/scream-vs-gcc-mahimahi-5g-ho-x0p33-snow-384x216.json` · Run: `python3 experiment.py scream-vs-gcc-mahimahi-5g-ho-x0p33-snow-384x216`

**Status:** 6 of 6 runs completed.

### `scream-vs-gcc-mahimahi-5g-rb-x0p33-snow-384x216`

SCReAM vs GCC on one fixed translated 5G Mahimahi RB-like trace scaled to 33% capacity, using calibrated high-entropy 384x216x30 snow video. The trace is fixed; the comparison varies only the congestion controller.


**Configurations:** `83` (scream), `84` (gcc) · **Reps:** 3

Spec: `specs/experiments/scream-vs-gcc-mahimahi-5g-rb-x0p33-snow-384x216.yaml` · Record: `runs/experiments/scream-vs-gcc-mahimahi-5g-rb-x0p33-snow-384x216.json` · Run: `python3 experiment.py scream-vs-gcc-mahimahi-5g-rb-x0p33-snow-384x216`

**Status:** 6 of 6 runs completed.


## Required metrics

- `frame_count`
- `encoder_target_kbps`
- `frame_latency`
- `wire_bytes`
- `encoded_bitrate`
- `decoder_errors`
- `late_drops`


## Reproducibility

This notebook is generated from `specs/hypotheses/h4.yaml` and `analysis/hypotheses/results/h4_report.json`. To regenerate:

```sh
python3 analysis/hypotheses/build_reports.py
python3 analysis/hypotheses/build_pages.py
python3 analysis/hypotheses/h4_scream_realistic_trace_advantage.py
```

Source: Project-internal 5G-recording SCReAM / GCC experiment
